# EcoData · S10 — Series de tiempo e inferencia en streaming
**Lanzeloth Vargas Marichi** · Proyecto Sello EcoData · Big Data, UPeU

La guía predice `CE(t+1)` con `lead()`, compara contra **persistencia**, elige el modelo ganador, lo guarda y lo aplica en vivo sin estado. En EcoData la variable es **PM2.5 a +1 hora** (Juliaca), con las **variables y el objetivo del proyecto** (`aire_comun`):

| Guía S10 | EcoData |
|---|---|
| Gold de S04 (`campo_electrico_particionado`) | histórico de Open-Meteo/CAMS de Juliaca (`aq_juliaca_2023.json`) |
| `label = lead("CE", 1)` | `label = lead("pm25", 1)` y objetivo del proyecto `delta = pm25(t+1) − pm25(t)`, verificando que la hora siguiente es consecutiva |
| 8 variables del mismo instante | las del proyecto calculables con **un solo evento**: `l0` (PM2.5), `pm10d` (PM10 − PM2.5), `hs` y `hc` (hora objetivo, seno/coseno), calculadas **dentro del Pipeline** |
| productor `campo-electrico-eventos` | topic **`ecodata-s10-aire`**, horas de la **cola del conjunto de prueba** publicadas a ritmo controlado |
| inferencia sin estado | idéntica: una lectura basta; no hay ventanas ni watermark |

> **Relación con el modelo de producción (S12):** el GBT de producción agrega 24 h de historial (8 variables: `l0, d1, d3, d24, m24, pm10d, hs, hc`) con la misma pérdida absoluta y el mismo objetivo `delta`: MAE 0,473 vs 0,687 de persistencia (−31,1 %). Este cuaderno aplica la mecánica de la guía —comparar, elegir al ganador, reentrenarlo, guardarlo y servirlo— con las variables que no necesitan estado.

## 0. Preparación y SparkSession

In [ ]:
import sys, json, time, threading, urllib.request, urllib.parse, os
from datetime import datetime, timezone
import pandas as pd
sys.path.insert(0, "/tmp/s12/scripts")
import aire_comun as aire
import openmeteo_a_kafka as pub_aire
from kafka import KafkaProducer, KafkaConsumer, TopicPartition
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

BOOTSTRAP = "kafka:9092"
PUSHGATEWAY = "http://pushgateway:9091"
PROMETHEUS_CANDIDATOS = ["http://prometheus:9090", "http://lambda26-prometheus:9090"]
PKG = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"

def crear_topic(nombre, particiones=3):
    adm = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
    try: adm.create_topics([NewTopic(nombre, num_partitions=particiones, replication_factor=1)]); print("topic creado:", nombre)
    except TopicAlreadyExistsError: print("el topic ya existía:", nombre)
    adm.close()

def prom(q):
    for base in PROMETHEUS_CANDIDATOS:
        try:
            u = f"{base}/api/v1/query?" + urllib.parse.urlencode({"query": q})
            return json.load(urllib.request.urlopen(u, timeout=4))["data"]["result"]
        except Exception:
            continue
    return None

def publicar_paced(topic, eventos, pausa_s=2.0):
    """Publica un evento cada pausa_s segundos en segundo plano (la celda no se bloquea)."""
    def run():
        p = KafkaProducer(bootstrap_servers=BOOTSTRAP, acks="all", key_serializer=str.encode,
                          value_serializer=lambda v: json.dumps(v).encode())
        for e in eventos:
            p.send(topic, key=e["sensorId"], value=e); p.flush(); time.sleep(pausa_s)
        p.close()
    th = threading.Thread(target=run, daemon=True); th.start(); return th

def desde_archivo(path="/tmp/s12/aq_juliaca_2023.json"):
    d = json.load(open(path)); h = d.get("hourly", d); out = []
    for t, a, b in zip(h["time"], h["pm2_5"], h["pm10"]):
        if a is None: continue
        ts = int(datetime.fromisoformat(t).replace(tzinfo=timezone.utc).timestamp() * 1000)
        out.append((ts, float(a), float(b) if b is not None else float(a)))
    return out

TOPIC = "ecodata-s10-aire"
crear_topic(TOPIC)

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, LongType
esquema_evento = (StructType().add("tipoEvento", StringType()).add("sensorId", StringType())
                  .add("pm25", DoubleType()).add("pm10", DoubleType())
                  .add("origen", StringType()).add("timestamp", LongType()))
spark = (SparkSession.builder.appName("ecodata-s10").master("local[2]")
         .config("spark.jars.packages", PKG)
         .config("spark.sql.shuffle.partitions", "3")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.port", "4051")          # distinto del 4040 que usan los jobs de S12
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)
from pyspark.sql.window import Window
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler, SQLTransformer
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

## 1. Business Understanding (3.1.2)
**Objetivo:** estimar el PM2.5 de la **hora siguiente** (`label = PM2.5 en t+1`) prediciendo `delta = PM2.5(t+1) − PM2.5(t)` a partir de la lectura actual (PM2.5, PM10 y hora objetivo).
**Alcance:** comparar persistencia, LinearRegression, RandomForest y GBT (pérdida absoluta, profundidades 3 y 4) con división cronológica y RMSE/MAE/R², elegir al ganador por MAE, reentrenarlo y aplicarlo en tiempo real.

## 2. Dataset supervisado con `lead()` y división cronológica (3.2–3.3)

In [ ]:
filas = desde_archivo()
df = spark.createDataFrame(filas, ["timestamp", "pm25", "pm10"])
orden = Window.orderBy("timestamp")
ds = (df.withColumn("label", F.lead("pm25", 1).over(orden))
        .withColumn("ts_sig", F.lead("timestamp", 1).over(orden)))
total = ds.count()
ds = ds.na.drop().filter(F.col("ts_sig") - F.col("timestamp") == 3600_000).drop("ts_sig")   # solo horas consecutivas
ds = ds.withColumn("delta", F.col("label") - F.col("pm25"))                                   # objetivo del proyecto: delta a +1 h
print(f"Filas históricas: {total:,} · filas del dataset supervisado: {ds.count():,} (se descartan la última y cualquier hueco)")
ds.orderBy("timestamp").show(5)

In [ ]:
num = ds.withColumn("n", F.row_number().over(orden))
corte = int(num.count() * 0.80)
train = num.filter(F.col("n") <= corte).drop("n"); test = num.filter(F.col("n") > corte).drop("n")
fmax_train = train.agg(F.max("timestamp")).first()[0]; fmin_test = test.agg(F.min("timestamp")).first()[0]
assert fmax_train < fmin_test, "La división no respeta el orden cronológico"
f = lambda ms: pd.to_datetime(ms, unit="ms")
print(f"Entrenamiento: {train.count():,} filas (hasta {f(fmax_train)} UTC)")
print(f"Prueba:        {test.count():,} filas (desde {f(fmin_test)} UTC) → sin fuga de información")

## 3. Baseline de persistencia, calculado ANTES de entrenar (3.4.1)
`PM2.5(t+1) = PM2.5(t)`: es lo que hay que superar (caso M4 de la guía).

In [ ]:
ev = {m: RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName=m) for m in ("rmse", "mae", "r2")}
def evaluar(pred, nombre):
    r = {"Modelo": nombre, "RMSE": ev["rmse"].evaluate(pred), "MAE": ev["mae"].evaluate(pred), "R2": ev["r2"].evaluate(pred)}
    print(f"{nombre}: RMSE={r['RMSE']:.4f}  MAE={r['MAE']:.4f}  R2={r['R2']:.4f}"); return r
def a_nivel(pred):
    """El modelo predice delta; el PM2.5 de la hora siguiente es pm25(t) + delta."""
    return pred.withColumn("prediction", F.col("pm25") + F.col("prediction"))
res_pers = evaluar(test.withColumn("prediction", F.col("pm25")), "Persistencia (PM2.5_t+1 = PM2.5_t)")

## 4. Modelado comparado (3.4.2–3.5.1)
Las variables del proyecto calculables con un solo evento (`l0`, `pm10d`, `hs`, `hc`) se calculan **dentro del Pipeline** (`SQLTransformer`), de modo que en producción el modelo solo necesita las columnas del evento. La hora objetivo es `(hora UTC + 1) mod 24`, igual que en `aire_comun.features`.

In [ ]:
HORA = SQLTransformer(statement="""SELECT *,
    pm25 AS l0,
    pm10 - pm25 AS pm10d,
    sin(2 * pi() * ((hour(CAST(`timestamp` / 1000 AS TIMESTAMP)) + 1) % 24) / 24) AS hs,
    cos(2 * pi() * ((hour(CAST(`timestamp` / 1000 AS TIMESTAMP)) + 1) % 24) / 24) AS hc FROM __THIS__""")
ENS = VectorAssembler(inputCols=["l0", "pm10d", "hs", "hc"], outputCol="features")
candidatos = {
    "LinearRegression": LinearRegression(featuresCol="features", labelCol="delta"),
    "RandomForestRegressor": RandomForestRegressor(featuresCol="features", labelCol="delta", numTrees=50, maxDepth=8, seed=42),
    "GBT profundidad 3 (pérdida absoluta)": GBTRegressor(featuresCol="features", labelCol="delta", lossType="absolute", maxIter=80, maxDepth=3, stepSize=0.1, seed=7),
    "GBT profundidad 4 (pérdida absoluta)": GBTRegressor(featuresCol="features", labelCol="delta", lossType="absolute", maxIter=80, maxDepth=4, stepSize=0.1, seed=7),
}
modelos, resultados = {}, [res_pers]
for nombre, est in candidatos.items():
    m = Pipeline(stages=[HORA, ENS, est]).fit(train); modelos[nombre] = m
    resultados.append(evaluar(a_nivel(m.transform(test)), nombre))
comp = pd.DataFrame(resultados)
comp["Mejora RMSE vs persistencia %"] = 100 * (res_pers["RMSE"] - comp["RMSE"]) / res_pers["RMSE"]
comp["Mejora MAE vs persistencia %"] = 100 * (res_pers["MAE"] - comp["MAE"]) / res_pers["MAE"]
comp.round(4)

## 5. Selección del modelo (3.5.2)

In [ ]:
mej = comp.iloc[1:].sort_values("MAE").iloc[0]
ganador = mej["Modelo"]; margen = mej["Mejora MAE vs persistencia %"]; margen_rmse = mej["Mejora RMSE vs persistencia %"]
print(f"Mejor modelo por MAE: {ganador} · mejora de MAE vs persistencia: {margen:+.2f} % · de RMSE: {margen_rmse:+.2f} %")
if margen <= 0:
    print("LECTURA: ningún modelo superó a la persistencia en MAE → no se justifica desplegar; se guarda igual para la infraestructura (como pide la guía).")
elif margen < 3:
    print("LECTURA: el margen es mínimo → probablemente no justifica el costo de mantener un modelo entrenado; su valor está en la infraestructura de inferencia.")
else:
    print("LECTURA: el margen es apreciable → el modelo se justifica frente a la persistencia, con la salvedad de validar el error en producción (Regla #37, S11).")

## 6. Reentrenar el ganador, guardar y recargar el PipelineModel (3.6.1)
El ganador se **reentrena** con su configuración sobre el conjunto de entrenamiento y se comprueba que reproduce el MAE de la comparación. Se guarda el **Pipeline completo** (variables + ensamblado + modelo) y se verifica que recargado predice lo mismo.

In [ ]:
est_ganador = candidatos[ganador].copy()
modelo_final = Pipeline(stages=[HORA, ENS, est_ganador]).fit(train)          # reentrenamiento del ganador
mae_final = evaluar(a_nivel(modelo_final.transform(test)), f"{ganador} (reentrenado)")["MAE"]
assert abs(mae_final - mej["MAE"]) < 1e-6, "El reentrenamiento no reproduce el MAE de la comparación"
modelos[ganador] = modelo_final
RUTA_MODELO = "./artifacts/models/modelo_pm25_s10"
modelo_final.write().overwrite().save(RUTA_MODELO)
cargado = PipelineModel.load(RUTA_MODELO)
a = modelo_final.transform(test).select("timestamp", F.col("prediction").alias("antes"))
b = cargado.transform(test).select("timestamp", F.col("prediction").alias("despues"))
dif = a.join(b, "timestamp").agg(F.max(F.abs(F.col("antes") - F.col("despues")))).first()[0]
print("Modelo guardado y recargado en:", RUTA_MODELO)
print("Diferencia máxima entre predicciones antes/después de recargar:", dif)
assert dif < 1e-9

## 7. Inferencia en tiempo real, sin estado (3.7)
Se publican, a ritmo controlado, las últimas 40 horas **reales del conjunto de prueba**; el modelo recargado predice cada lectura tal como llega (sin ventanas ni watermark), y como las horas son consecutivas, la predicción de cada lectura se puede comparar con la hora real siguiente.

In [ ]:
crudo = (spark.readStream.format("kafka").option("kafka.bootstrap.servers", BOOTSTRAP)
         .option("subscribe", "ecodata-s10-aire").option("startingOffsets", "latest").option("failOnDataLoss", "false").load())
eventos = (crudo.select("topic", "partition", "offset", F.col("timestamp").alias("kafkaTimestamp"),
                        F.from_json(F.col("value").cast("string"), esquema_evento).alias("e"))
           .select("topic", "partition", "offset", "kafkaTimestamp", "e.*"))
predicciones_stream = cargado.transform(eventos)          # el Pipeline guardado hace todo el trabajo
RUTA_PRED = "./artifacts/s10_predicciones_stream"; CHK_PRED = "./artifacts/chk_s10"
captura = []
def guardar(dfm, bid):
    n = dfm.count()
    if n == 0:
        captura.append(dict(batchId=bid, filas=0)); return
    out = dfm.select(F.lit(bid).alias("batchId"), "sensorId", "timestamp", "pm25", (F.col("pm25") + F.col("prediction")).alias("pred_pm25_siguiente")   # pm25(t) + delta)
    out.write.mode("append").parquet(RUTA_PRED)
    for r in out.collect(): captura.append({**r.asDict(), "filas": n})

if "q" in dir() and q.isActive: q.stop()
q = predicciones_stream.writeStream.foreachBatch(guardar).option("checkpointLocation", CHK_PRED).start()
print("consulta activa:", q.isActive)

In [ ]:
cola = test.orderBy(F.desc("timestamp")).limit(40).orderBy("timestamp").select("timestamp", "pm25", "pm10").collect()
eventos_pub = [aire.evento(r["timestamp"], r["pm25"], r["pm10"], "openmeteo-replay") for r in cola]
h = publicar_paced(TOPIC, eventos_pub, pausa_s=3.0)
print(len(eventos_pub), "eventos publicados a razón de 1 cada 3 s (≈ 2 min)")

In [ ]:
time.sleep(5)
print("micro-lotes con datos hasta ahora:", len([c for c in captura if c.get("filas")]))
for c in captura[-5:]: print(c)          # repite esta celda mientras corre la publicación

In [ ]:
h.join(timeout=180); time.sleep(10)
q.stop()
res = spark.read.parquet(RUTA_PRED).orderBy("timestamp").toPandas()
res["pm25_real_siguiente"] = res["pm25"].shift(-1)
aud = res.dropna()
mae_prod = (aud["pred_pm25_siguiente"] - aud["pm25_real_siguiente"]).abs().mean()
mae_pers = (aud["pm25"] - aud["pm25_real_siguiente"]).abs().mean()
print(f"Predicciones capturadas: {len(res)} · comparables: {len(aud)}")
print(f"MAE del modelo en el stream: {mae_prod:.4f} µg/m³ · MAE de persistencia en las mismas horas: {mae_pers:.4f} µg/m³")
res.head(8)

In [ ]:
ax = res.set_index(pd.to_datetime(res.timestamp, unit="ms"))[["pm25", "pred_pm25_siguiente"]].plot(
        figsize=(11, 3.5), marker="o", title="PM2.5 observado y predicción de la hora siguiente (stream)")
ax.set_ylabel("µg/m³")

## 8. Hallazgo para el informe
Escribe un error o hallazgo **real** de tu ejecución. Candidatos que suelen aparecer: (a) `Micro-lote 0: sin filas` (no es un error); (b) horas no consecutivas descartadas al construir `label`; (c) el modelo que «gana» por un margen mínimo o pierde frente a persistencia (caso M4); (d) `hour()` dependiente de la zona horaria: por eso `spark.sql.session.timeZone = UTC`.

### Qué capturar (con **reloj del sistema y tu perfil visibles**)
1. Sección 2: `lead()`, conteos y división cronológica con fechas.
2. Sección 3: baseline. Sección 4: tabla comparativa. Sección 5: selección.
3. Sección 6: guardado/recarga y diferencia ≈ 0.
4. Sección 7: bitácora del stream, tabla de predicciones y MAE; Kafka UI → `ecodata-s10-aire`.

In [ ]:
spark.stop()